# 1. Context

Credit Risk Classification

The task is to flag prospects who are likely to default credit lines before the firm offers them the service. Our threshold for marking a given prospect is x on scale y.

The positive class is `bad/default-risk`. This equals the minority group we want to predict from our data, we want to find the prospects who are likely to default the credit line.

![image.png](attachment:image.png)

In [6]:
from pathlib import Path
import json
import sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data_contract.py').exists() and (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Could not locate the assignment project root')

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_contract import load_raw_data, make_xy, contract_summary
frame = load_raw_data(ROOT / 'data' / 'raw' / 'SouthGermanCredit.asc')
X, y = make_xy(frame)
contract_summary(frame)


{'rows': 1000,
 'columns': 21,
 'predictor_columns': 20,
 'missing_values': 0,
 'duplicate_rows': 0,
 'raw_target_counts': {'0': 300, '1': 700},
 'positive_target_counts': {'0': 700, '1': 300},
 'numeric_columns': ['laufzeit', 'hoehe', 'alter'],
 'categorical_columns': ['laufkont',
  'moral',
  'verw',
  'sparkont',
  'beszeit',
  'rate',
  'famges',
  'buerge',
  'wohnzeit',
  'verm',
  'weitkred',
  'wohn',
  'bishkred',
  'beruf',
  'pers',
  'telef',
  'gastarb']}

## 1b. Regenerate the analysis artifacts

Run this cell after a clean installation to rebuild Waves 0-5 from the immutable raw source. The frozen test is not used for selection.

In [7]:
from src.build_waves_0_3 import main as build_waves_0_3
from src.run_wave4 import main as run_wave4
from src.run_wave5 import main as run_wave5
build_waves_0_3()
run_wave4()
run_wave5()


{
  "status": "complete",
  "development_rows": 800,
  "test_rows": 200,
  "models": [
    "majority_baseline",
    "logistic_regression",
    "decision_tree",
    "random_forest"
  ],
  "provisional_selection": "random_forest",
  "artifacts": "C:\\Users\\katom\\iCloudDrive\\Smaller Creator\\Sky\\Marc Torrens Assignment\\artifacts"
}
{
  "status": "complete",
  "selected_model": "logistic_regression",
  "recommended_threshold": 0.5667006784529198,
  "recommended_precision_oof": 0.63125,
  "recommended_recall_oof": 0.42083333333333334,
  "recommended_f1_oof": 0.505,
  "recommended_accuracy_oof": 0.7525,
  "recommended_monthly_reviews_oof": 1600.0,
  "recommended_net_value_oof": 2635600.0
}
{
  "status": "complete",
  "model": "logistic_regression",
  "development": [
    {
      "scenario": "base",
      "sample_size": 800,
      "threshold": 0.5667006784529198,
      "tn": 501,
      "fp": 59,
      "fn": 139,
      "tp": 101,
      "accuracy": 0.7525,
      "precision": 0.63125,
     

In [8]:
oof = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-4-oof-threshold-table-all-models.csv')
test = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-4-selected-model-frozen-test-thresholds.csv')
oof[oof['policy'].eq('recommended_capacity_constrained')]


,model,policy,threshold,selection_rule,tn,fp,fn,tp,accuracy,precision,...,monthly_reviews,capacity_utilization,expected_defaults_prevented,gross_loss_avoided,monthly_review_cost,lost_contribution_from_friction,net_screening_value,cost_per_expected_default_prevented,missed_avoidable_loss,selected_count
5,majority_baseline,recommended_capacity_constrained,0.300000,top_k,453,107,187,53,0.6325,0.33125,...,1600.0,1.0,238.5,1431000.0,56000.0,64200.0,1310800.0,234.800839,5049000.0,160
11,logistic_regression,recommended_capacity_constrained,0.566701,top_k,501,59,139,101,0.7525,0.63125,...,1600.0,1.0,454.5,2727000.0,56000.0,35400.0,2635600.0,123.212321,3753000.0,160
17,decision_tree,recommended_capacity_constrained,0.508021,top_k,482,78,158,82,0.7050,0.51250,...,1600.0,1.0,369.0,2214000.0,56000.0,46800.0,2111200.0,151.761518,4266000.0,160
23,random_forest,recommended_capacity_constrained,0.459926,top_k,497,63,143,97,0.7425,0.60625,...,1600.0,1.0,436.5,2619000.0,56000.0,37800.0,2525200.0,128.293242,3861000.0,160


## 2. Classification evaluation

Report accuracy, precision, recall and F1 together. Accuracy is secondary because the review queue is capacity-constrained. The threshold policy is selected on development out-of-fold predictions and confirmed once on the frozen test set.

In [9]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'monthly_reviews', 'net_screening_value']
selected_model = json.loads((ROOT / 'artifacts' / 'manifests' / 'wave-4-policy.json').read_text())['selected_model']
oof[oof['model'].eq(selected_model)][['policy', 'threshold'] + [m for m in metrics if m in oof.columns]]


,policy,threshold,accuracy,precision,recall,f1,monthly_reviews,net_screening_value
6,threshold_0_50,0.500000,0.74375,0.587940,0.487500,0.533030,1990.0,3040150.0
7,high_recall,0.220000,0.67000,0.469849,0.779167,0.586207,3980.0,4783100.0
8,high_precision,0.800000,0.73500,0.804348,0.154167,0.258741,460.0,977500.0
9,capacity_top_20_percent,0.566701,0.75250,0.631250,0.420833,0.505000,1600.0,2635600.0
10,max_development_net_value,0.020000,0.35750,0.317819,0.995833,0.481855,7520.0,5882000.0
11,recommended_capacity_constrained,0.566701,0.75250,0.631250,0.420833,0.505000,1600.0,2635600.0


In [10]:
sensitivity = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-5-sensitivity-development.csv')
sensitivity[['scenario', 'net_screening_value', 'cost_per_expected_default_prevented']]


,scenario,net_screening_value,cost_per_expected_default_prevented
0,base,2635600.0,123.212321
1,conservative,1083360.0,237.623762
2,adverse,439200.0,475.247525
3,upside,4786760.0,66.006601


## 3. Recommendation and limitations

Use the selected model only as a ranking aid for the exact top 20% manual-review queue. Do not auto-approve or auto-reject. The model and policy remain subject to calibration, subgroup review, performance-improvement experiments and independent review.